In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer
import torch

/home/misha/Desktop/knowledge_distillation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
class StudentModel:
    def __init__(self, model_name: str, input_txt: str):
        self.model = self._initialize_model(model_name=model_name)
        self.tokenizer = self._initialize_tokenizer(model_name=model_name)
        self.input_tokens = self._tokenize_input(input_txt=input_txt)
    
    def _initialize_model(self, model_name: str):
        return AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=model_name,
                                                    device_map='auto',
                                                    dtype='auto')
    
    def _initialize_tokenizer(self, model_name: str):
        return AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)
    
    def _tokenize_input(self, input_txt: str):
        return self.tokenizer(input_txt, return_tensors='pt')
    
    def forward_pass(self):
        with torch.no_grad():
            outputs = self.model(**self.input_tokens)
        
        return outputs

In [7]:
st_model = StudentModel('Qwen/Qwen3-0.6B', "Mr. President, welcome!")

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 14276.96it/s]


In [8]:
import torch.nn.functional as F

for idx, logits in enumerate(st_model.forward_pass().logits[0]):
    print(f'{idx} — {logits}')
    probs = F.softmax(logits, dim=-1)
    sorted_probs, index = torch.sort(probs, descending=True)
    for prob, idx in zip(sorted_probs[:5], index[:5]):
        print(f'{st_model.tokenizer.convert_ids_to_tokens(int(idx))} — {prob}')
    print('###############################')

0 — tensor([5.4688, 4.4375, 5.0625,  ..., 2.2031, 2.2031, 2.2031],
       dtype=torch.bfloat16)
Question — 0.0255126953125
ĠAnswer — 0.02392578125
ĠQuestion — 0.0093994140625
ĠInstructions — 0.0093994140625
est — 0.007781982421875
###############################
1 — tensor([ 1.7344,  2.4844, -2.0625,  ..., -2.5312, -2.5312, -2.5312],
       dtype=torch.bfloat16)
ĠJohnson — 0.10791015625
ĠSmith — 0.07421875
ĠThompson — 0.045166015625
ĠLee — 0.022705078125
ĠJones — 0.0166015625
###############################
2 — tensor([10.7500,  9.0625,  3.5156,  ..., -1.4609, -1.4609, -1.4609],
       dtype=torch.bfloat16)
, — 0.87109375
's — 0.021728515625
Ġand — 0.01092529296875
Ġsaid — 0.00909423828125
: — 0.005859375
###############################
3 — tensor([ 4.1250,  4.0938,  1.4453,  ..., -1.5547, -1.5547, -1.5547],
       dtype=torch.bfloat16)
ĠI — 0.1328125
Ġin — 0.058837890625
Ġthe — 0.0458984375
Ġyou — 0.0458984375
Ġwhat — 0.0296630859375
###############################
4 — tensor([15.3750

In [2]:
student_name = 'Qwen/Qwen3-0.6B'
student_model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=student_name,
                                                     device_map='auto',
                                                     dtype='auto')
student_tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=student_name)

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 16777.86it/s]


In [30]:
type(student_tokenizer)

transformers.models.qwen2.tokenization_qwen2.Qwen2Tokenizer

In [52]:
student_tokenizer("Mr. President, welcome!")

{'input_ids': [12275, 13, 4795, 11, 10565, 0], 'attention_mask': [1, 1, 1, 1, 1, 1]}

In [3]:
prompt = "What is x equal to in 25x + 50 = 125?"

input_ids = student_tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    return_tensors="pt",
    tokenize=True,
    do_sample=False,
    enable_thinking=False,
    add_generation_prompt=True
)['input_ids'].to(student_model.device)

In [4]:
streamer = TextStreamer(student_tokenizer, skip_prompt=True, skip_special_tokens=True)

In [5]:
output = student_model.generate(
    input_ids,
    do_sample=True,
    temperature=0.1,
    top_k=50,
    repetition_penalty=1.05,
    max_new_tokens=512,
    streamer=streamer,
    output_logits=True,
    return_dict_in_generate=True, 
)

We are given the equation:

$$
25x + 50 = 125
$$

### Step 1: Subtract 50 from both sides

$$
25x = 125 - 50
$$
$$
25x = 75
$$

### Step 2: Divide both sides by 25

$$
x = \frac{75}{25}
$$
$$
x = 3
$$

### ✅ Final Answer:
$$
\boxed{x = 3}
$$


In [6]:
input_ids = student_tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    return_tensors="pt",
    tokenize=True,
    do_sample=False,
    enable_thinking=False,
    add_generation_prompt=True
)

In [7]:
import torch.nn.functional as F

probs = [F.softmax(p, dim=-1) for p in output.logits]
tokens = [token for token in output.sequences]

In [8]:
input_ids

{'input_ids': tensor([[151644,    872,    198,   3838,    374,    856,   6144,    311,    304,
            220,     17,     20,     87,    488,    220,     20,     15,    284,
            220,     16,     17,     20,     30, 151645,    198, 151644,  77091,
            198, 151667,    271, 151668,    271]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1]])}

In [9]:
print(f'Число токенов промпта — {len(input_ids['input_ids'][0])}')
print(f'Число логитов — {len(output.logits)}')
print(f'Число токенов на выходе (вместе с промптом) — {len(output.sequences[0])}')

Число токенов промпта — 32
Число логитов — 119
Число токенов на выходе (вместе с промптом) — 151


In [ ]:
import torch

for idx, logits in zip(output.sequences[0][len(input_ids['input_ids'][0]):], output.logits):
    print(f'{idx} — {student_tokenizer.decode(idx)}')
    probs = F.softmax(logits[0], dim=-1)
    sorted_probs, index = torch.sort(probs, descending=True)
    for prob, idx in zip(sorted_probs[:5], index[:5]):
        print(f'{student_tokenizer.convert_ids_to_tokens(int(idx))} — {prob}')
    print('###############################')

1654 — We
We — 0.9455896019935608
To — 0.05334651470184326
Given — 0.0005229908274486661
Start — 0.00021801500406581908
Let — 0.00016979027714114636
###############################
525 —  are
Ġare — 0.9866431951522827
Ġstart — 0.005866795312613249
Ġneed — 0.0031402690801769495
Ġwant — 0.0014833579771220684
're — 0.0011552403448149562
###############################
2661 —  given
Ġgiven — 0.9752706289291382
Ġsolving — 0.02024109475314617
Ġasked — 0.0035173750948160887
Ġtold — 0.0006926122005097568
Ġlooking — 0.00010621552064549178
###############################
279 —  the
Ġthe — 0.9955249428749084
:ĊĊ — 0.0021777020301669836
Ġan — 0.0016959960339590907
Ġa — 0.0002947199100162834
Ġthis — 0.000202557843294926
###############################
23606 —  equation
Ġequation — 0.9922459721565247
Ġlinear — 0.007575891446322203
Ġinequality — 3.975462823291309e-05
Ġalgebra — 3.975462823291309e-05
Ġfollowing — 3.508333975332789e-05
###############################
1447 — :


:ĊĊ — 0.9847478270530701

In [41]:
transition_scores = student_model.compute_transition_scores(
    output.sequences, output.scores, normalize_logits=True
)

In [ ]:
transition_scoresinput_length = inputs.input_ids.shape[1]
generated_tokens = output.sequences[:, input_length:]
for tok, score in zip(generated_tokens[0], transition_scores[0]):
    # | token | token string | logits | probability
    print(f"| {tok:5d} | {tokenizer.decode(tok):8s} | {score.numpy():.4f} | {np.exp(score.numpy()):.2%}")

NameError: name 'inputs' is not defined

In [ ]:
teacher_name = 'openbmb/MiniCPM5-1B'
teacher_model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=teacher_name,
                                                     device_map='auto',
                                                     dtype='bfloat16')
teacher_tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=teacher_name)